In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
movies = pd.read_csv("movie.csv")
ratings = pd.read_csv("rating.csv")

In [3]:
movies["genres"] = movies["genres"].fillna("")

In [4]:
movie_ratings = ratings.groupby("movieId")["rating"].agg(
    ["mean", "count"]
).reset_index()

In [5]:
movie_ratings.columns = [
    "movieId",
    "avg_rating",
    "rating_count"
]

movies = movies.merge(
    movie_ratings,
    on="movieId",
    how="left"
)

In [6]:
movies["avg_rating"] = movies["avg_rating"].fillna(0)
movies["rating_count"] = movies["rating_count"].fillna(0)

In [7]:
tfidf = TfidfVectorizer()

tfidf_matrix = tfidf.fit_transform(
    movies["genres"]
)
similarity_matrix = cosine_similarity(
    tfidf_matrix,
    tfidf_matrix
)

In [8]:
movie_index = pd.Series(
    movies.index,
    index=movies["title"]
).drop_duplicates()


In [9]:

def recommend_movies(movie_name, top_n=10):

    if movie_name not in movie_index:
        print("Movie not found.")
        return

    idx = movie_index[movie_name]

    similarity_scores = list(
        enumerate(similarity_matrix[idx])
    )

    recommendations = []

    for movie_idx, sim_score in similarity_scores:

        avg_rating = movies.iloc[movie_idx]["avg_rating"]

        rating_count = movies.iloc[movie_idx]["rating_count"]

        # Hybrid Score
        final_score = (
            sim_score * 0.7
            +
            (avg_rating / 5.0) * 0.3
        )

        recommendations.append(
            (
                movie_idx,
                final_score,
                rating_count
            )
        )

    recommendations = sorted(
        recommendations,
        key=lambda x: x[1],
        reverse=True
    )

    recommendations = recommendations[1:top_n+1]

    print("\nRecommended Movies\n")

    for movie_idx, score, count in recommendations:

        print(
            f"Title : {movies.iloc[movie_idx]['title']}"
        )

        print(
            f"Genre : {movies.iloc[movie_idx]['genres']}"
        )

        print(
            f"Average Rating : "
            f"{movies.iloc[movie_idx]['avg_rating']:.2f}"
        )

        print(
            f"Number of Ratings : {int(count)}"
        )

        print(
            f"Recommendation Score : "
            f"{score:.3f}"
        )

        print("-" * 50)


In [10]:
while True:

    movie_name = input(
        "\nEnter Movie Name (type exit to stop): "
    )

    if movie_name.lower() == "exit":
        print("Program Stopped.")
        break

    recommend_movies(movie_name)


Enter Movie Name (type exit to stop):  exit


Program Stopped.
